# 06 A 1-D CNN finds a DNA motif

Lecture 11, architectures and representations. Book chapter 9 (convolutions
for sequences).

**Goals**

- Encode DNA as a one-hot tensor of shape `(N, 4, L)` and train a small
  convolutional network: `Conv1d` → ReLU → global max pooling → `Linear`.
- Read the learned filter as a position weight matrix, and see where in a
  sequence it fires.
- Compare with two string kernels of lecture 5, the spectrum kernel (a
  linear SVM on $k$-mer counts) and the mismatch kernel (which also counts
  $k$-mers with up to $m$ differences), when the planted motif carries point
  mutations.

**Expected findings** (asserted in `tests/test_examples.py::test_06_*`):

1. With the exact motif, the CNN reaches more than 95 % test accuracy, and
   its most important filter fires at a fixed offset from the planted motif
   in more than 90 % of the positive sequences.
2. With 15 % point mutations in the motif, the $(6, 1)$-mismatch kernel is
   better than the best spectrum kernel ($k = 3, 4, 5$), and the CNN is
   better than both.

The data come from `dna_motifs.py` in this folder.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dna_motifs import kmer_counts, make_motif_dataset, mismatch_counts, one_hot
from sklearn.svm import SVC

import glassnn.functional as F
from glassnn import Tensor, manual_seed, nn, no_grad, optim
from glassnn.data import DataLoader

MOTIF = "TGACGTCA"

## Data

2000 random sequences of length 60; about half contain the motif
`TGACGTCA` at a random position. The first 1000 are for training.

In [ ]:
def split(mutation_rate):
    sequences, labels, positions = make_motif_dataset(
        2000, motif=MOTIF, mutation_rate=mutation_rate, seed=1
    )
    return (
        (sequences[:1000], labels[:1000]),
        (sequences[1000:], labels[1000:]),
        positions[1000:],
    )


(train_s, train_y), (test_s, test_y), test_positions = split(0.0)
print(train_s[0], "label", train_y[0])
print("one-hot shape:", one_hot(train_s).shape)

## The model

A filter of width 8 slides along the sequence and computes, at every
position, a weighted sum of the bases in the window: a learned position
weight matrix. Global max pooling keeps the best match of each filter
anywhere in the sequence, so the prediction does not depend on where the
motif is.

In [ ]:
def train_cnn(sequences, labels, seed=0, epochs=30):
    X = one_hot(sequences)
    manual_seed(seed)
    model = nn.Sequential(
        nn.Conv1d(4, 16, kernel_size=8),
        nn.ReLU(),
        nn.MaxPool1d(X.shape[2] - 8 + 1),  # global max pooling
        nn.Flatten(),
        nn.Linear(16, 2),
    )
    optimizer = optim.Adam(model.parameters(), lr=0.003)
    for _ in range(epochs):
        for xb, yb in DataLoader(X, labels, batch_size=32, shuffle=True):
            optimizer.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            optimizer.step()
    return model


def accuracy(model, sequences, labels):
    with no_grad():
        logits = model(Tensor(one_hot(sequences))).data
    return float((logits.argmax(axis=1) == labels).mean())


model = train_cnn(train_s, train_y)
print(model)
print("test accuracy:", accuracy(model, test_s, test_y))

## What did the network learn?

The filter whose output pushes most towards "motif present" is the one with
the largest difference between the two rows of the last layer. Its weights,
one row per base, form a position weight matrix; the base with the largest
weight at each position is the consensus.

In [ ]:
votes = model[4].weight.data[1] - model[4].weight.data[0]
best = int(np.argmax(votes))
W = model[0].weight.data[best]  # (4, 8)
consensus = "".join("ACGT"[i] for i in W.argmax(axis=0))
print("filter", best, "consensus:", consensus, "| motif:", MOTIF)

fig, ax = plt.subplots(figsize=(6, 2.5))
image = ax.imshow(W, cmap="RdBu_r", vmin=-np.abs(W).max(), vmax=np.abs(W).max())
ax.set_yticks(range(4), list("ACGT"))
ax.set_xlabel("position in the filter")
fig.colorbar(image)
plt.show()

In [ ]:
positives = np.flatnonzero(test_y == 1)
with no_grad():
    responses = model[0](Tensor(one_hot([test_s[i] for i in positives]))).data[:, best]
offsets = responses.argmax(axis=1) - test_positions[positives]
values, counts = np.unique(offsets, return_counts=True)
print(
    "offset of the strongest response from the motif start:",
    dict(zip(values.tolist(), counts.tolist(), strict=True)),
)

i = positives[0]
plt.figure(figsize=(7, 2.5))
plt.plot(responses[0])
plt.axvline(test_positions[i], color="gray", linestyle=":", label="planted motif")
plt.xlabel("window start")
plt.ylabel(f"response of filter {best}")
plt.legend()
plt.show()

The filter may be shifted by a few positions with respect to the motif (it
then sees 6 or 7 of its 8 bases, plus a random neighbour): what matters is
that it fires at the *same* offset in almost every positive sequence.

## Comparison with string kernels

The spectrum kernel of lecture 5 compares two sequences by their $k$-mer
counts. With the exact motif, a long enough $k$ sees it directly. With
point mutations, most copies of the motif no longer contain any given
$k$-mer, while a convolutional filter still gives a high score to a window
that matches 7 of 8 bases. The $(k, m)$-mismatch kernel counts every
$k$-mer also for all $k$-mers within Hamming distance $m$ of it, so it
tolerates some mutations too. Both kernels use the best of a few values of
$C$ *on the test set*, which favours them a little.

In [ ]:
def best_spectrum(train_s, train_y, test_s, test_y):
    scores = {}
    for k in (3, 4, 5):
        A, B = kmer_counts(train_s, k), kmer_counts(test_s, k)
        scores[k] = max(
            SVC(kernel="linear", C=C).fit(A, train_y).score(B, test_y)
            for C in (0.001, 0.01, 0.1)
        )
    return max(scores.values())


def mismatch(train_s, train_y, test_s, test_y, k=6, m=1):
    A, B = mismatch_counts(train_s, k, m), mismatch_counts(test_s, k, m)
    return max(
        SVC(kernel="linear", C=C).fit(A, train_y).score(B, test_y)
        for C in (0.0001, 0.001, 0.01)
    )


rates = [0.0, 0.1, 0.15, 0.2, 0.3]
scores = {
    "CNN (mean of 3 seeds)": [],
    "mismatch kernel (6, 1)": [],
    "best spectrum kernel": [],
}
for rate in rates:
    (tr_s, tr_y), (te_s, te_y), _ = split(rate)
    cnn = np.mean(
        [accuracy(train_cnn(tr_s, tr_y, seed=s), te_s, te_y) for s in range(3)]
    )
    row = [cnn, mismatch(tr_s, tr_y, te_s, te_y), best_spectrum(tr_s, tr_y, te_s, te_y)]
    for name, value in zip(scores, row, strict=True):
        scores[name].append(value)
    print(
        f"mutation rate {rate:.2f}: CNN {row[0]:.3f} | "
        f"mismatch {row[1]:.3f} | spectrum {row[2]:.3f}"
    )

plt.figure(figsize=(6, 3.5))
for (name, values), style in zip(scores.items(), ["o-", "^-", "s-"], strict=True):
    plt.plot(rates, values, style, label=name)
plt.xlabel("mutation rate per motif base")
plt.ylabel("test accuracy")
plt.legend()
plt.show()

**Findings.**

1. With the exact motif all three methods are nearly perfect; the CNN
   filter learns the motif (up to a shift) and fires at a fixed offset from
   it.
2. With 10 to 20 % mutations the CNN is better by about 6 points than the
   spectrum kernel, because a filter scores partial matches while the
   spectrum kernel needs exact $k$-mer matches. The mismatch kernel, which
   tolerates one difference per 6-mer, recovers a quarter to a third of
   that gap. At 30 % mutations the signal fades, and the three methods are
   within 2 points of each other (the mismatch kernel slightly ahead).

The comparison still favours the CNN a little: its filter width (8) equals
the motif length, while the kernels' $k$ is chosen from a few small values.

**Exercises.**

1. Use filters of width 12 and 4. Which accuracy and which consensus do you
   get?
2. Replace the global max pooling by global *average* pooling
   (`nn.AvgPool1d`). Why does it work worse for a single motif per
   sequence?
3. Plant the reverse complement of a non-palindromic motif in half of the
   positives. How many filters does the network need?